
# Exploratory Data Analysis & Preprocessing 


## Imports

In [106]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.impute import SimpleImputer
from scipy import sparse

import warnings
warnings.filterwarnings("ignore")


## Load

In [107]:
df = pd.read_csv("../data/raw/rents.csv")
print(df.shape)
df.head(3)


(11657, 8)


,address,district,area,bedrooms,garage,type,rent,total
0,Rua Herval,Belenzinho,21,1,0,Studio e kitnet,2400,2939
1,Avenida São Miguel,Vila Marieta,15,1,1,Studio e kitnet,1030,1345
2,Rua Oscar Freire,Pinheiros,18,1,0,Apartamento,4000,4661


## Target and feature columns

In [108]:
num_cols = ["area", "bedrooms", "garage"]
cat_cols = ["address", "district", "type"]

y = df.iloc[:, -1].values
X = df.loc[:, num_cols + cat_cols]



print('Numeric cols:', num_cols)
print('Categorical cols:', cat_cols)


Numeric cols: ['area', 'bedrooms', 'garage']
Categorical cols: ['address', 'district', 'type']


## Stratify by price tiers to preserve distribution of the target in train/test

In [109]:
q = pd.qcut(y, q=5, duplicates='drop')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=q
)
print(X_train.shape, X_test.shape)


(9325, 6) (2332, 6)


# Helper functions

#

In [110]:
def clip_upper(arr, upper):
    if sparse.issparse(arr):
        arr = arr.toarray()
    return np.clip(arr, None, upper)

def to_dense_if_sparse(Xm):
    return Xm.toarray() if sparse.issparse(Xm) else Xm

def normalize_text_series(s: pd.Series) -> pd.Series:
    s = s.copy()
    mask = s.notna()
    s.loc[mask] = s.loc[mask].astype(str).str.strip().str.lower()
    return s

def normalize_text_df(df_in: pd.DataFrame) -> pd.DataFrame:
    df_out = df_in.copy()
    for c in df_out.columns:
        df_out[c] = normalize_text_series(df_out[c])
    return df_out


## Area p99 clipping

In [111]:
if "area" in X_train.columns:
    p99_area = pd.to_numeric(X_train["area"], errors='coerce').quantile(0.99)
    print('p99_area =', float(p99_area))
else:
    p99_area = None
    print(f"[info] Column '{"area"}' not in data; skipping area clipping.")


p99_area = 387.28000000000065


# Build pipelines

In [112]:
num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')) 
])

cat_pipe = Pipeline(steps=[
    ('normalizer', FunctionTransformer(normalize_text_df)), 
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))  
])


area_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('clipper', FunctionTransformer(lambda x: clip_upper(x, upper=p99_area))),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipe, [c for c in num_cols if c != 'area']),
        ('cat', cat_pipe, cat_cols),
        ('area', area_pipe, ['area']),
    ],
    remainder='drop' 
)

## Fit/transform

In [113]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


has_nan_train = np.isnan(to_dense_if_sparse(X_train_processed)).any()
has_nan_test = np.isnan(to_dense_if_sparse(X_test_processed)).any()

print(f'Has NaNs in train? {has_nan_train}')
print(f'Has NaNs in test? {has_nan_test}')

Has NaNs in train? False
Has NaNs in test? False


## Save splits

In [114]:
train_df = X_train.copy()
test_df  = X_test.copy()

train_df["total"] = y_train
test_df["total"]  = y_test

output_dir = Path("../data/work/")
output_dir.mkdir(parents=True, exist_ok=True)
train_path = output_dir / "train.csv"
test_path  = output_dir / "test.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Arquivo de treino salvo em: {train_path}")
print(f"Arquivo de teste salvo em:  {test_path}")

Arquivo de treino salvo em: ..\data\work\train.csv
Arquivo de teste salvo em:  ..\data\work\test.csv
